# VAE V2 — KL Warm-up Training and Final Comparison
This notebook preserves V1 and trains V2 with linear beta annealing for epochs 1–10.

In [ ]:
import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
!pip install -q kagglehub scikit-image torchmetrics torch-fidelity
from pathlib import Path
import os, subprocess, sys
PROJECT_ROOT = Path('/content/Digital-Evidence-GenAI')
if not PROJECT_ROOT.exists():
    subprocess.run(['git','clone','https://github.com/chetanraje27/Digital-Evidence-GenAI.git',str(PROJECT_ROOT)], check=True)
sys.path.insert(0, str(PROJECT_ROOT/'src'))
print('Project:', PROJECT_ROOT)

In [ ]:
# Download CASIA only when the manifest target is unavailable.
import csv, kagglehub
first = next(csv.DictReader(open(PROJECT_ROOT/'data/splits/train.csv', encoding='utf-8')))
expected = PROJECT_ROOT / first['image_path']
if not expected.exists():
    downloaded = Path(kagglehub.dataset_download('divg07/casia-20-image-tampering-detection-dataset'))
    casia2 = next(downloaded.rglob('CASIA2'))
    target = PROJECT_ROOT/'data/raw/CASIA2'
    target.parent.mkdir(parents=True, exist_ok=True)
    if not target.exists(): os.symlink(casia2, target, target_is_directory=True)
print('Dataset ready:', expected.exists())

In [ ]:
# Full V2 training: writes only V2 paths and never overwrites V1.
from argparse import Namespace
from train_vae_v2 import train
args = Namespace(
    splits_dir=PROJECT_ROOT/'data/splits', checkpoint_path=PROJECT_ROOT/'checkpoints/best_vae_v2.pth',
    history_path=PROJECT_ROOT/'results/vae_v2_training_history.csv',
    curve_path=PROJECT_ROOT/'outputs/vae/vae_v2_loss_curves.png',
    summary_path=PROJECT_ROOT/'results/vae_v2_training_summary.json',
    image_size=128, batch_size=32, num_workers=2, latent_dim=128,
    learning_rate=0.0005, max_epochs=50, patience=5, target_beta=0.001,
    warmup_epochs=10, seed=42, smoke_test=False, smoke_batches=2)
training_summary = train(args)
training_summary

In [ ]:
# Complete V2 test evaluation and standard 2048-feature FID.
from evaluate_vae_v2 import evaluate
eval_args = Namespace(
    splits_dir=PROJECT_ROOT/'data/splits', v1_checkpoint=PROJECT_ROOT/'checkpoints/best_vae.pth',
    v2_checkpoint=PROJECT_ROOT/'checkpoints/best_vae_v2.pth',
    v1_metrics=PROJECT_ROOT/'results/vae_test_metrics.json',
    metrics_path=PROJECT_ROOT/'results/vae_v2_test_metrics.json',
    per_image_path=PROJECT_ROOT/'results/vae_v2_test_per_image_metrics.csv',
    comparison_path=PROJECT_ROOT/'results/vae_v1_vs_v2_comparison.csv',
    grid_path=PROJECT_ROOT/'outputs/vae/vae_v1_vs_v2_reconstruction.png',
    image_size=128, batch_size=32, num_workers=2, latent_dim=128, beta=0.001,
    seed=42, expected_test_count=1892, grid_per_class=3)
comparison = evaluate(eval_args)
comparison

In [ ]:
v1, v2 = comparison['v1'], comparison['v2']
print('V1:')
print('MSE', v1['mse']); print('PSNR', v1['psnr']); print('SSIM', v1['ssim']); print('FID', v1['fid'])
print('\nV2:')
print('MSE', v2['mse']); print('PSNR', v2['psnr']); print('SSIM', v2['ssim']); print('FID', v2['fid']); print('KL', v2['kl'])
print('\nBest epoch', training_summary['best_epoch'])
print('Training time', training_summary['training_time_seconds'])
recon_improved = v2['mse'] < v1['mse'] and v2['psnr'] > v1['psnr'] and v2['ssim'] > v1['ssim']
generation_improved = v2['fid'] < v1['fid']
print('V2 improved reconstruction:', recon_improved)
print('V2 improved generation:', generation_improved)